# Jaccard Similarity: Original vs Successfully Adapted Puzzles

Compare each **successful/equivalent** adapted puzzle to its original. Preprocessing removes the shared task instruction and discards `<context>` and all subsequent text. Three word-set Jaccard variants are reported. Jaccard measures lexical overlap, **not** semantic equivalence; equivalence is determined by the audit label.

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
OUTPUT_DIR = Path('/content/jaccard_audit_results') if Path('/content').exists() else Path('jaccard_audit_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV = Path('/content/Sheet1_copy-Grid_view.csv')
if not INPUT_CSV.exists():
    INPUT_CSV = Path('Sheet1_copy-Grid_view.csv')
if not INPUT_CSV.exists():
    raise FileNotFoundError('Upload Sheet1_copy-Grid_view.csv or update INPUT_CSV.')

df = pd.read_csv(INPUT_CSV)
print(f'Loaded {len(df)} audit rows')

## 1. Keep only successful/equivalent adaptations

In [ ]:
required = ['original_sample', 'final_adapted_puzzle', 'corrected_outcome']
missing = [x for x in required if x not in df.columns]
if missing:
    raise KeyError(f'Missing required columns: {missing}')

success_df = df.loc[
    df['corrected_outcome'].astype(str).str.strip().str.lower().eq('equivalent')
].copy()
print(f'Successfully adapted pairs: {len(success_df)} of {len(df)} audit rows')

## 2. Preprocess both original and adapted puzzle text

In [ ]:
INSTRUCTION = re.compile(
    r'Your\s+task\s+is\s+to\s+provide\s+the\s+correct\s+answer\s+to\s+the\s+given\s+puzzle\s+using\s+the\s+clues\s*\.\s*',
    flags=re.IGNORECASE,
)
CONTEXT_OPEN = re.compile(r'<\s*context\b[^>]*>', flags=re.IGNORECASE)

def preprocess_puzzle(text):
    """Return puzzle text before <context>, without the common instruction.

    Missing values become empty strings. Handles case/whitespace differences
    in the context tag and instruction; collapses whitespace afterward.
    """
    if pd.isna(text):
        return ''
    text = str(text)
    text = CONTEXT_OPEN.split(text, maxsplit=1)[0]
    text = INSTRUCTION.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()

success_df['original_clean'] = success_df['original_sample'].apply(preprocess_puzzle)
success_df['adapted_clean'] = success_df['final_adapted_puzzle'].apply(preprocess_puzzle)

before = len(success_df)
success_df = success_df.loc[
    success_df['original_clean'].ne('') & success_df['adapted_clean'].ne('')
].copy()
print(f'Nonempty successful pairs after preprocessing: {len(success_df)} / {before}')
if success_df.empty:
    raise ValueError('No nonempty successful pairs remain. Check the tag placement and data columns.')
display(success_df[['original_clean', 'adapted_clean']].head(3))

## 3. Jaccard metrics

In [ ]:
def tok_basic(x):
    return set(str(x).lower().split())

def tok_normalized(x):
    return set(re.findall(r'\b[a-z0-9]+\b', str(x).lower()))

def jac(a, b):
    return np.nan if not a and not b else len(a & b) / len(a | b)

def text_jac(a, b, tokenizer):
    return jac(tokenizer(a), tokenizer(b))

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOP = set(stopwords.words('english'))

def tok_content(x):
    return {w for w in tok_normalized(x) if w not in STOP and len(w) > 1}

for name, tokenizer in [
    ('jaccard_basic', tok_basic),
    ('jaccard_normalized', tok_normalized),
    ('jaccard_content', tok_content),
]:
    success_df[name] = success_df.apply(
        lambda r: text_jac(r['original_clean'], r['adapted_clean'], tokenizer), axis=1
    )

metrics = ['jaccard_basic', 'jaccard_normalized', 'jaccard_content']
display(success_df[metrics].head())

## 4. Summary statistics — successful pairs only

In [ ]:
summary = success_df[metrics].agg(['count', 'mean', 'median', 'std', 'min', 'max']).T
summary['q25'] = success_df[metrics].quantile(0.25)
summary['q75'] = success_df[metrics].quantile(0.75)
display(summary.round(4))
summary.to_csv(OUTPUT_DIR / 'jaccard_success_summary.csv')

## 5. Bootstrap 95% confidence intervals for mean Jaccard

These confidence intervals describe the mean **within successful adaptations**. No failed-group comparison or significance test is performed.

In [ ]:
rng = np.random.default_rng(SEED)
def bootstrap_mean_ci(values, B=10000):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return (np.nan, np.nan, np.nan)
    means = np.array([rng.choice(values, size=len(values), replace=True).mean() for _ in range(B)])
    lo, hi = np.percentile(means, [2.5, 97.5])
    return (values.mean(), lo, hi)

ci_rows = []
for metric in metrics:
    mean, lo, hi = bootstrap_mean_ci(success_df[metric].to_numpy())
    ci_rows.append({'metric': metric, 'n': success_df[metric].notna().sum(),
                    'mean': mean, 'CI_95_low': lo, 'CI_95_high': hi})
ci_df = pd.DataFrame(ci_rows)
display(ci_df.round(4))
ci_df.to_csv(OUTPUT_DIR / 'jaccard_success_mean_ci.csv', index=False)

## 6. Distribution of successful-pair normalized Jaccard

In [ ]:
s = success_df['jaccard_normalized'].dropna().to_numpy()
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.hist(s, bins=np.linspace(0, 1, 21), edgecolor='white')
ax.axvline(np.mean(s), color='black', linestyle='--', linewidth=1.5,
           label=f'Mean = {np.mean(s):.3f}')
ax.axvline(np.median(s), color='gray', linestyle=':', linewidth=1.5,
           label=f'Median = {np.median(s):.3f}')
ax.set(xlim=(0, 1), xlabel='Normalized word-set Jaccard', ylabel='Number of pairs',
       title=f'Original vs successful adaptations (n={len(s)})')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'jaccard_success_distribution.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'jaccard_success_distribution.pdf', bbox_inches='tight')
plt.show()

## 7. Inspect lowest- and highest-overlap successful pairs

In [ ]:
inspect_cols = [x for x in ['Puzzle_Number', 'Question_Number', 'jaccard_normalized',
    'original_clean', 'adapted_clean'] if x in success_df.columns]
print('Lowest overlap:')
display(success_df.nsmallest(10, 'jaccard_normalized')[inspect_cols])
print('Highest overlap:')
display(success_df.nlargest(10, 'jaccard_normalized')[inspect_cols])

## 8. Export sample-level successful results

In [ ]:
export_cols = [x for x in ['Puzzle_Number', 'Question_Number', 'corrected_outcome',
    'original_clean', 'adapted_clean', *metrics] if x in success_df.columns]
success_df[export_cols].to_csv(OUTPUT_DIR / 'jaccard_success_sample_level.csv', index=False)
print('Saved results in', OUTPUT_DIR.resolve())

## Paper reporting template

> **Lexical-overlap analysis.** For **[N]** successfully adapted, independently audited equivalent puzzles, we removed the common task instruction and excluded text beginning with the `<context>` tag before calculating word-set Jaccard between each original puzzle and its adaptation. The mean normalized Jaccard was **[MEAN]** (SD **[SD]**, bootstrap 95% CI **[LOW, HIGH]**), and the mean content-word Jaccard was **[CONTENT MEAN]**. These values characterize lexical modification within successful adaptations; they do not measure semantic equivalence.

In [ ]:
# Optional Colab download of the generated results.
# Run only in Google Colab if you want a ZIP download.
# import shutil
# from google.colab import files
# archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', str(OUTPUT_DIR))
# files.download(archive)